# 05 · Consultar SQL e documentos existentes

In [1]:
import os, sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
from pprint import pprint
from src.config import DATA

<a id="ferramentas"></a>

| Ferramenta | Tipo | Operação |
|---|---|---|
| buscar_vagas | API externa | Remotive e Arbeitnow normalizam anúncios e preservam a fonte |
| converter_moeda | API externa e regra fixa | AwesomeAPI fornece a taxa; Decimal calcula BRL |
| consultar_candidaturas | Text-to-query e SQLite | A pergunta vira consulta de leitura; a visão é limitada à sessão |
| recuperar_contexto | Busca semântica | FAISS carrega o índice do notebook 01 |
| avaliar_fit | Regra fixa | Interseção das habilidades explícitas |
| verify | Semântica e regras | NLI compara anúncio e afirmação; regras bloqueiam restrições explícitas |

FastAPI expõe o chat e uma rota de registro de candidatura. A escrita é explícita e
parametrizada; o executor de SQL gerado não pode escrever.

In [2]:
from tempfile import TemporaryDirectory
from unittest.mock import patch
from fastapi.testclient import TestClient
from src.agente import Radar
from src.api import app
question = 'Quais vagas acima de 5k USD eu ainda não apliquei?'
with TemporaryDirectory() as temp:
    radar = Radar(db_path=Path(temp) / 'vagas.db')
    with patch('src.api.get_radar', return_value=radar):
        client = TestClient(app)
        client.post('/chat', json={'session_id':'s1','message':'Mostre vagas de Python'}).raise_for_status()
        recorded = client.post('/candidaturas', json={'session_id':'s1','vaga_id':'demo-aurora','status':'enviada'})
        print('Registro manual:', recorded.raise_for_status().json())
    if os.getenv('NOTEBOOK_OPENAI') == '1':
        from src.config import live_model
        radar.llm = live_model()
    query, rows = radar._sql(question, 's1')
    print(question, '\n', query)
    pprint(rows)

/var/folders/fp/5_zj8d4n2g1fr35b3rkfs9c40000gn/T/ipykernel_11481/3779234617.py:8: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  radar = Radar(db_path=Path(temp) / 'vagas.db')
<string>:3: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
tool=buscar_vagas source=Remotive status=dummy


tool=buscar_vagas source=Arbeitnow status=dummy


/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1112 23:06:34.478000 11481 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


/Users/fernandotrinta/Desktop/Infnet/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 106/106 [00:00<00:00, 20354.17it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18272.03it/s]

Registro manual: {'session_id': 's1', 'vaga_id': 'demo-aurora', 'status': 'enviada'}


Quais vagas acima de 5k USD eu ainda não apliquei? 
 SELECT v."title", v."company"
FROM vagas AS v
LEFT JOIN candidaturas AS c
    ON c."vaga_id" = v."id"
WHERE v."salary_min" > 5000
  AND v."currency" = 'USD'
  AND (c."status" IS NULL OR c."status" <> 'enviada')
LIMIT 10;
[{'company': 'Atlas Systems', 'title': 'Senior Python Developer'}]


In [3]:
from src.demo import recovery_demo
result = recovery_demo()
pprint(result)

{'antes': [{'id': 'demo-moon',
            'kind': 'vaga',
            'source': 'Dados fictícios',
            'trecho': 'Python Software Engineer\n'
                      'Remote position. Python and Django. Salary: USD 8,000. '
                      'No further details are provided.',
            'url': 'https://example.com/jobs/moon'},
           {'id': 'demo-atlas',
            'kind': 'vaga',
            'source': 'Dados fictícios',
            'trecho': 'Senior Python Developer\n'
                      'Remote, US only. Python, FastAPI, PostgreSQL and '
                      'Docker. Salary: USD 9,000 per month.',
            'url': 'https://example.com/jobs/atlas'}],
 'consulta_original': 'desenvolvedor backend sênior',
 'depois': [{'id': 'demo-aurora',
             'kind': 'vaga',
             'source': 'Dados fictícios',
             'trecho': 'Senior Python Backend Engineer\n'
                       'Worldwide remote, including Brazil. Senior Python, '
                      

No conjunto avaliado, Aurora ficou fora do top 2. A consulta original
        privilegiou outros anúncios de software Python sem o cargo backend esperado.
        Reescrever para Senior Backend Engineer aproximou a consulta da terminologia
        do título da Aurora sem alterar o índice, os documentos ou o valor de k.

## Evidência

- **4.1** → [ferramentas](#ferramentas): Catálogo com descrição e tipo.
- **4.2** → [ferramentas](#ferramentas): Registro HTTP interno na célula SQL; HTTP externo no 02.
- **4.3** → [sql](#sql): Pergunta, consulta SQL e linha retornada.
- **4.4** → [recuperacao](#recuperacao): Executor chama recuperação sobre o índice salvo.
- **4.5** → [recuperacao](#recuperacao): Antes e depois com os próprios documentos recuperados.